Roll No.: 10

Name: Shubham Khairnar

Date of Performance: 25 Feb 2026

Problem Statement: Develop a model-based RL algorithm, such as Monte Carlo Tree Search (MCTS), to solve a
complex environment like Atari games.

In [ ]:
!pip install gymnasium[atari] ale-py autorom opencv-python numpy

In [ ]:
!AutoROM --accept-license

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis2.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/backgammon.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/bank_heist.bin
Inst

In [ ]:
import gymnasium as gym
import numpy as np
import cv2
import random
import math

In [ ]:
def process_frame(frame):
    frame = cv2.resize(frame, (84, 84))
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return frame / 255.0

In [ ]:
class Node:
    def __init__(self, state=None, parent=None):
        self.state = state
        self.parent = parent
        self.children = {}
        self.visits = 0
        self.value = 0

    def best_child(self, c=1.4):
        best_score = -float("inf")
        best = None

        for action, child in self.children.items():
            exploit = child.value / (child.visits + 1e-5)
            explore = math.sqrt(math.log(self.visits + 1) / (child.visits + 1e-5))
            score = exploit + c * explore

            if score > best_score:
                best_score = score
                best = child

        return best

In [ ]:
class MCTS:
    def __init__(self, env_name, simulations=20, max_depth=15):
        self.env_name = env_name
        self.simulations = simulations
        self.max_depth = max_depth

    def search(self):
        root = Node()

        for _ in range(self.simulations):
            env = gym.make(self.env_name)
            state, _ = env.reset()

            node = root

            # Selection
            while node.children:
                node = node.best_child()

            # Expansion
            if node.visits > 0:
                for action in range(env.action_space.n):
                    env.reset()
                    next_state, reward, terminated, truncated, _ = env.step(action)
                    node.children[action] = Node(next_state, node)

                node = random.choice(list(node.children.values()))

            # Simulation (Improved rollout)
            total_reward = 0
            done = False
            depth = 0

            while not done and depth < self.max_depth:
                action = 1 if random.random() > 0.3 else 0

                _, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                total_reward += reward
                depth += 1

            # Backpropagation
            while node is not None:
                node.visits += 1
                node.value += total_reward
                node = node.parent

        best_action = max(root.children.items(), key=lambda x: x[1].visits)[0]
        return best_action

In [ ]:
env = gym.make("CartPole-v1", render_mode="human")
mcts = MCTS("CartPole-v1", simulations=100, max_depth=30)

state, _ = env.reset()
done = False
total_reward = 0

while not done:
    action = mcts.search()
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward

print("Final Score:", total_reward)
env.close()

Final Score: 112.0
